# Multimodal Training: CLIP and LLaVA

This notebook demonstrates training vision-language models:
- **CLIP**: Image-text contrastive learning
- **LLaVA**: Vision-language instruction following

## What You'll Learn
1. How to process multimodal data (images + text)
2. Training CLIP for image-text alignment
3. Training LLaVA for image captioning/VQA
4. Evaluating with CLIP Score
5. Generating captions from images

## Prerequisites
- Understanding of transformers and fine-tuning
- Basic knowledge of computer vision
- Completed SFT notebook (01_understanding_sft.ipynb)

---

## 1. Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from transformers import CLIPProcessor, AutoProcessor

# Project imports
from src.models.vision_language import create_vision_language_model
from src.data.processors.multimodal import MultimodalDataProcessor
from src.data.collators.multimodal import create_multimodal_collator
from src.evaluation.metrics.multimodal import CLIPScoreMetric, ImageTextRetrievalMetric

print(f"✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Multimodal Data Processing

First, let's explore how multimodal data is structured and processed.

In [ ]:
# Create data processor
processor = MultimodalDataProcessor()

# Generate synthetic data for quick testing
print("Generating synthetic multimodal data...")
examples = processor.create_synthetic_data(num_examples=10)

print(f"\n✓ Created {len(examples)} examples")
print(f"\nExample structure:")
ex = examples[0]
print(f"  - Image: {type(ex.image)} {ex.image.size}")
print(f"  - Caption: {ex.caption}")
print(f"  - Text: {ex.text}")

In [ ]:
# Visualize some examples
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, ex in enumerate(examples[:6]):
    axes[i].imshow(ex.image)
    axes[i].set_title(ex.caption, fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle("Synthetic Multimodal Data", fontsize=14, y=1.02)
plt.show()

print("These are simple colored shapes - perfect for quick testing!")

## 3. Training CLIP

CLIP learns to align images and text in a shared embedding space.

### Key Concepts:
- **Contrastive Learning**: Maximize similarity for matching pairs, minimize for non-matching
- **Dual Encoders**: Separate vision and text encoders
- **Zero-Shot**: Can classify images with text prompts (no task-specific training)

### Architecture:
```
Image → Vision Encoder → Image Embedding ─┐
                                          ├─→ Cosine Similarity → Loss
Text  → Text Encoder   → Text Embedding  ─┘
```

In [ ]:
# Load CLIP model
print("Loading CLIP model...")
clip_model = create_vision_language_model(
    model_type="clip",
    model_name="openai/clip-vit-base-patch32",
    use_lora=True,  # Use LoRA for efficient training
    lora_config={
        'r': 8,
        'lora_alpha': 16,
        'target_modules': ['q_proj', 'v_proj'],
    },
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"\n✓ CLIP model loaded")
print(f"Model type: {type(clip_model)}")
print(f"Device: {clip_model.device}")

# Load processor
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print(f"✓ Processor loaded")

In [ ]:
# Test CLIP on synthetic data
print("Testing CLIP image-text alignment...\n")

test_image = examples[0].image
test_texts = [
    examples[0].caption,  # Correct caption
    examples[1].caption,  # Wrong caption
    "A random unrelated caption",
]

# Compute similarities
similarities = clip_model.compute_similarity(
    images=[test_image] * len(test_texts),
    texts=test_texts,
)

print(f"Image: {examples[0].caption}\n")
for text, sim in zip(test_texts, similarities):
    print(f"Text: '{text}'")
    print(f"Similarity: {sim.item():.3f}\n")

print("Higher similarity = better match!")

### Training CLIP (Quick Demo)

For full training, use `scripts/train/train_multimodal.py` with Hydra configs.

In [ ]:
from transformers import TrainingArguments, Trainer
from datasets import Dataset

# Prepare dataset
train_examples = processor.create_synthetic_data(num_examples=50)
train_data = {
    'image': [ex.image for ex in train_examples],
    'text': [ex.caption for ex in train_examples],
}
train_dataset = Dataset.from_dict(train_data)

# Create data collator
collator = create_multimodal_collator(
    model_type="clip",
    tokenizer=clip_processor.tokenizer,
    image_processor=clip_processor.image_processor,
)

# Training arguments (minimal for demo)
training_args = TrainingArguments(
    output_dir="./outputs/clip_demo",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=1e-5,
    logging_steps=5,
    remove_unused_columns=False,  # Important!
    report_to=[],  # Disable wandb for demo
)

# Create trainer
trainer = Trainer(
    model=clip_model.model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)

print("Ready to train! (Uncomment next line to run)")
# trainer.train()  # Uncomment to actually train

## 4. Training LLaVA

LLaVA combines a vision encoder (CLIP) with a language model (LLaMA/Vicuna) for multimodal instruction following.

### Architecture:
```
Image → CLIP Vision Encoder → Visual Features ─┐
                                               ├─→ LLaMA → Response
Instruction → Tokenizer → Text Tokens ─────────┘
```

### Training:
- Vision encoder is usually **frozen** (pre-trained CLIP)
- Language model is **fine-tuned** on image+instruction pairs
- LoRA + 4-bit quantization makes 7B models trainable on consumer GPUs

In [ ]:
# Note: LLaVA requires ~8GB GPU with 4-bit quantization
# For CPU/small GPU, skip this section or use smaller model

print("Loading LLaVA model...")
print("Note: This requires significant memory. Set use_4bit=True for 7B model.")

# Example configuration (adjust for your hardware)
llava_config = {
    'model_type': 'llava',
    'model_name': 'llava-hf/llava-1.5-7b-hf',
    'use_lora': True,
    'lora_config': {
        'r': 16,
        'lora_alpha': 32,
        'target_modules': ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    },
    'use_4bit': True,  # Essential for 7B model
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

# Uncomment to load (requires GPU with 8GB+ VRAM)
# llava_model = create_vision_language_model(**llava_config)
# print("✓ LLaVA model loaded")

print("\n(Skipped loading for demo - uncomment to load actual model)")

In [ ]:
# Example: Generate captions with LLaVA
# (Requires loaded model from above)

"""
# Generate captions
test_images = [ex.image for ex in examples[:3]]
prompts = ["Describe this image in detail:"] * 3

captions = llava_model.generate(
    images=test_images,
    prompts=prompts,
    max_new_tokens=50,
    temperature=0.7,
)

# Display results
for img, prompt, caption in zip(test_images, prompts, captions):
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.text(0.1, 0.5, f"Prompt: {prompt}\n\nGenerated:\n{caption}",
             fontsize=10, verticalalignment='center')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
"""

print("See above for generation example code.")
print("For full LLaVA training, use: python scripts/train/train_multimodal.py experiment=llava_instruction")

## 5. Evaluation with CLIP Score

CLIP Score measures how well text matches an image using CLIP embeddings.

### How it works:
1. Encode image → image embedding
2. Encode text → text embedding
3. Compute cosine similarity
4. Scale to 0-100

Higher score = better alignment!

In [ ]:
# Create CLIP Score metric
print("Initializing CLIP Score metric...")
clip_metric = CLIPScoreMetric(
    model_name="openai/clip-vit-base-patch32",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print("✓ Ready")

In [ ]:
# Evaluate on synthetic data
test_examples = processor.create_synthetic_data(num_examples=20)
images = [ex.image for ex in test_examples]
captions = [ex.caption for ex in test_examples]

print("Computing CLIP Scores...\n")
scores = clip_metric.compute(images, captions)

print("Results:")
print(f"  Average CLIP Score: {scores['clip_score']:.2f}")
print(f"  Std Dev: {scores['clip_score_std']:.2f}")
print(f"  Range: [{scores['clip_score_min']:.2f}, {scores['clip_score_max']:.2f}]")

In [ ]:
# Visualize scores
individual_scores = scores['individual_scores']

plt.figure(figsize=(10, 4))

# Histogram
plt.subplot(1, 2, 1)
plt.hist(individual_scores, bins=15, edgecolor='black', alpha=0.7)
plt.xlabel('CLIP Score')
plt.ylabel('Count')
plt.title('Distribution of CLIP Scores')
plt.axvline(scores['clip_score'], color='red', linestyle='--', label='Mean')
plt.legend()

# Per-example
plt.subplot(1, 2, 2)
plt.plot(individual_scores, marker='o', linestyle='-', alpha=0.7)
plt.xlabel('Example Index')
plt.ylabel('CLIP Score')
plt.title('Per-Example Scores')
plt.axhline(scores['clip_score'], color='red', linestyle='--', label='Mean')
plt.legend()

plt.tight_layout()
plt.show()

## 6. Image-Text Retrieval

Retrieval metrics measure how well the model can:
- Find the correct image for a text query (Text-to-Image)
- Find the correct text for an image (Image-to-Text)

Metrics: **Recall@K** (is correct match in top-K results?)

In [ ]:
# Create retrieval metric
print("Initializing retrieval metric...")
retrieval_metric = ImageTextRetrievalMetric(
    model_name="openai/clip-vit-base-patch32",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print("✓ Ready")

In [ ]:
# Compute retrieval metrics
print("Computing retrieval metrics...\n")
retrieval_scores = retrieval_metric.compute(
    images=images,
    texts=captions,
    k_values=[1, 5, 10],
)

print("Text-to-Image Retrieval:")
print(f"  R@1:  {retrieval_scores['t2i_recall@1']:.1%}")
print(f"  R@5:  {retrieval_scores['t2i_recall@5']:.1%}")
print(f"  R@10: {retrieval_scores['t2i_recall@10']:.1%}")

print("\nImage-to-Text Retrieval:")
print(f"  R@1:  {retrieval_scores['i2t_recall@1']:.1%}")
print(f"  R@5:  {retrieval_scores['i2t_recall@5']:.1%}")
print(f"  R@10: {retrieval_scores['i2t_recall@10']:.1%}")

In [ ]:
# Visualize retrieval results
import pandas as pd

results_df = pd.DataFrame([
    {
        'Metric': 'Text→Image',
        'R@1': retrieval_scores['t2i_recall@1'],
        'R@5': retrieval_scores['t2i_recall@5'],
        'R@10': retrieval_scores['t2i_recall@10'],
    },
    {
        'Metric': 'Image→Text',
        'R@1': retrieval_scores['i2t_recall@1'],
        'R@5': retrieval_scores['i2t_recall@5'],
        'R@10': retrieval_scores['i2t_recall@10'],
    },
])

# Plot
results_df.set_index('Metric').plot(kind='bar', figsize=(8, 5), width=0.7)
plt.ylabel('Recall')
plt.title('Retrieval Performance')
plt.ylim(0, 1.1)
plt.legend(title='K')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Summary and Next Steps

### What We Covered:
1. ✓ Multimodal data processing (images + text)
2. ✓ CLIP training and image-text alignment
3. ✓ LLaVA architecture for instruction following
4. ✓ Evaluation with CLIP Score
5. ✓ Retrieval metrics (Recall@K)

### Key Takeaways:
- **CLIP**: Learns joint embeddings for images and text (contrastive learning)
- **LLaVA**: Combines vision + language for instruction following
- **CLIP Score**: Automatic metric for image-caption quality
- **LoRA + 4-bit**: Makes large models trainable on consumer hardware

### Production Training:
For real datasets and full training, use the command-line scripts:

```bash
# Train CLIP on COCO Captions
python scripts/train/train_multimodal.py \
    experiment=clip_image_caption \
    data.dataset_name=coco \
    data.max_train_samples=50000

# Train LLaVA on instruction data
python scripts/train/train_multimodal.py \
    experiment=llava_instruction \
    data.dataset_name=coco \
    data.use_instruction_format=true

# Evaluate trained model
python scripts/evaluate/evaluate_multimodal.py \
    --model_path ./outputs/clip_caption \
    --model_type clip \
    --dataset coco \
    --num_examples 1000
```

### Further Exploration:
1. Try real datasets: COCO Captions, Flickr30k, Conceptual Captions
2. Experiment with different LoRA ranks
3. Compare CLIP variants (ViT-B/32 vs ViT-L/14)
4. Fine-tune for specific domains (medical images, satellite imagery, etc.)
5. Extend to video-language models (CLIP + temporal attention)

### Next Notebooks:
- **07_ppo_rlhf_deep_dive.ipynb**: RLHF with multimodal reward models
- **09_comparing_techniques.ipynb**: Compare multimodal vs text-only training